# CIC-IDS-2017 feature preparation

This notebook starts from the cleaned dataset produced by `01_data_exploration.ipynb`. Its first stage removes identifier columns that would encourage memorization of the CIC-IDS-2017 laboratory, separates the remaining features from the multiclass target, and creates a reproducible stratified train/test split. No scaling, encoding, correlation filtering, resampling or learned feature selection is performed before the split.

## 1. Imports and paths

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

processed_dataset_candidates = [
    Path("../data/processed/cicids2017_cleaned.parquet"),
    Path("ml/data/processed/cicids2017_cleaned.parquet"),
]

PROCESSED_DATA_PATH = next(
    (
        path.resolve()
        for path in processed_dataset_candidates
        if path.exists()
    ),
    None,
)

if PROCESSED_DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find cicids2017_cleaned.parquet. "
        "Run 01_data_exploration.ipynb first."
    )

RANDOM_STATE = 42
TEST_SIZE = 0.20

print(f"Processed dataset: {PROCESSED_DATA_PATH}")
print(f"File size: {PROCESSED_DATA_PATH.stat().st_size / (1024 ** 2):,.2f} MiB")
print(f"Random state: {RANDOM_STATE}")
print(f"Test fraction: {TEST_SIZE:.0%}")

Processed dataset: C:\Users\ademz\Desktop\9raya\DoS-Intrusion-Detection-System\ml\data\processed\cicids2017_cleaned.parquet
File size: 360.48 MiB
Random state: 42
Test fraction: 20%


## 2. Load and validate the cleaned data

Confirm that the processed file matches the final dimensions and target structure recorded in the exploration notebook.

In [2]:
data = pd.read_parquet(PROCESSED_DATA_PATH)

EXPECTED_ROWS = 2_824_752
EXPECTED_COLUMNS = 76
EXPECTED_LABELS = 15

dataset_validation = pd.DataFrame(
    {
        "Observed": [
            data.shape[0],
            data.shape[1],
            data["Label"].nunique(dropna=False)
            if "Label" in data.columns
            else 0,
            int(data["Label"].isna().sum())
            if "Label" in data.columns
            else data.shape[0],
        ],
        "Expected": [
            EXPECTED_ROWS,
            EXPECTED_COLUMNS,
            EXPECTED_LABELS,
            0,
        ],
    },
    index=[
        "Rows",
        "Columns",
        "Detailed target classes",
        "Missing target labels",
    ],
)
display(dataset_validation)

if not dataset_validation["Observed"].equals(
    dataset_validation["Expected"]
):
    raise AssertionError(
        "The cleaned dataset does not match the expected EDA output."
    )

print("The cleaned dataset matches the final EDA output.")

,Observed,Expected
Rows,2824752,2824752
Columns,76,76
Detailed target classes,15,15
Missing target labels,0,0


The cleaned dataset matches the final EDA output.


## 3. Review all cleaned columns

Display the complete cleaned schema before excluding identifiers or separating the target.

In [3]:
cleaned_column_overview = pd.DataFrame(
    {
        "Position": range(1, len(data.columns) + 1),
        "Column": data.columns,
        "Data type": data.dtypes.astype(str).to_numpy(),
    }
).set_index("Position")

with pd.option_context("display.max_rows", None):
    display(cleaned_column_overview)
print(f"Total cleaned columns: {len(cleaned_column_overview)}")

,Column,Data type
Position,,
1,Flow ID,object
2,Src IP,object
3,Src Port,float64
4,Dst IP,object
5,Dst Port,float64
6,Protocol,float64
7,Timestamp,object
8,Flow Duration,float64
9,Total Fwd Packet,float64


Total cleaned columns: 76


## 4. Remove identifier and leakage-prone columns

`Flow ID`, endpoint IP addresses and `Timestamp` describe this particular laboratory capture rather than transferable flow behavior. These fixed schema exclusions do not depend on statistics calculated from the dataset, so remove them directly from the cleaned table before separating features and target. Ports and `Protocol` are retained for now.

In [4]:
identifier_columns = [
    "Flow ID",
    "Src IP",
    "Dst IP",
    "Timestamp",
]
missing_identifier_columns = [
    column for column in identifier_columns if column not in data.columns
]
if missing_identifier_columns:
    raise KeyError(
        "Expected identifier columns are missing: "
        f"{missing_identifier_columns}"
    )

identifier_decisions = pd.DataFrame(
    {
        "Column": identifier_columns,
        "Reason for exclusion": [
            "Constructed flow identifier; encourages record memorization",
            "Source identity is specific to the CIC-IDS-2017 laboratory",
            "Destination identity is specific to the CIC-IDS-2017 laboratory",
            "Encodes the published attack schedule and capture day",
        ],
    }
)
display(identifier_decisions)

data.drop(columns=identifier_columns, inplace=True)
print(f"Columns remaining after fixed exclusions: {data.shape[1]}")

,Column,Reason for exclusion
0,Flow ID,Constructed flow identifier; encourages record...
1,Src IP,Source identity is specific to the CIC-IDS-201...
2,Dst IP,Destination identity is specific to the CIC-ID...
3,Timestamp,Encodes the published attack schedule and capt...


Columns remaining after fixed exclusions: 72


## 5. Separate features and target

Keep the original 15-class `Label` as the target. Using `pop` separates it without duplicating the large feature table in memory.

In [5]:
y = data.pop("Label")
X = data
del data

print(f"Feature rows: {X.shape[0]:,}")
print(f"Feature columns: {X.shape[1]}")
print(f"Target rows: {len(y):,}")
print(f"Target classes: {y.nunique()}")

Feature rows: 2,824,752
Feature columns: 71
Target rows: 2,824,752
Target classes: 15


## 6. Inspect Protocol values

Start by displaying the unmodified `Protocol.value_counts()` result. Assign readable names only after the values actually present in the cleaned dataset are visible. The stored `Protocol` column is not changed in this section.

In [6]:
raw_protocol_counts = (
    X["Protocol"].value_counts(dropna=False).sort_index()
)
raw_protocol_distribution = (
    raw_protocol_counts
    .rename_axis("Raw Protocol value")
    .rename("Flow count")
    .to_frame()
)
raw_protocol_distribution["Percentage of dataset"] = (
    raw_protocol_distribution["Flow count"] / len(X) * 100
)
display(raw_protocol_distribution)

,Flow count,Percentage of dataset
Raw Protocol value,,
0.0,1696,0.060041
6.0,1823631,64.558977
17.0,999425,35.380982


The observed protocol values and their names are:

- `0`: HOPOPT
- `6`: TCP
- `17`: UDP

The notebook leaves the `Protocol` column unchanged. `Label` is the target rather than a feature, and the representation of source and destination ports remains a later feature-engineering decision.

## 7. Create the train/test split

Use a reproducible stratified 80/20 split so every target class is represented in both subsets. This split estimates performance on held-out flows from the same CIC-IDS-2017 capture; it does not by itself demonstrate generalization to a different network or time period. All later data-dependent preprocessing must be fitted using `X_train` and `y_train` only.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

if not X_train.index.intersection(X_test.index).empty:
    raise AssertionError("Training and test rows overlap.")

del X, y

split_size_summary = pd.DataFrame(
    {
        "Rows": [len(X_train), len(X_test)],
        "Percentage of dataset": [
            len(X_train) / (len(X_train) + len(X_test)) * 100,
            len(X_test) / (len(X_train) + len(X_test)) * 100,
        ],
    },
    index=["Training set", "Test set"],
)
display(split_size_summary)
print(f"Features in each split: {X_train.shape[1]}")

,Rows,Percentage of dataset
Training set,2259801,79.999979
Test set,564951,20.000021


Features in each split: 71


## 8. Verify split distributions

Confirm that no class disappeared and quantify how closely stratification preserved the original multiclass distribution.

In [8]:
training_label_counts = y_train.value_counts()
test_label_counts = y_test.value_counts()
total_label_counts = (
    training_label_counts.add(test_label_counts, fill_value=0)
    .astype("int64")
    .sort_values(ascending=False)
)

split_label_distribution = pd.DataFrame(
    {
        "Total flows": total_label_counts,
        "Training flows": training_label_counts.reindex(
            total_label_counts.index, fill_value=0
        ),
        "Test flows": test_label_counts.reindex(
            total_label_counts.index, fill_value=0
        ),
    }
)
split_label_distribution["Training percentage"] = (
    split_label_distribution["Training flows"] / len(y_train) * 100
)
split_label_distribution["Test percentage"] = (
    split_label_distribution["Test flows"] / len(y_test) * 100
)
split_label_distribution["Absolute difference (percentage points)"] = (
    split_label_distribution["Training percentage"]
    .sub(split_label_distribution["Test percentage"])
    .abs()
)
display(split_label_distribution)

if split_label_distribution[["Training flows", "Test flows"]].eq(0).any().any():
    raise AssertionError("At least one target class is absent from a split.")

if not (
    split_label_distribution["Training flows"]
    + split_label_distribution["Test flows"]
).equals(split_label_distribution["Total flows"]):
    raise AssertionError("Split label counts do not reproduce the full target.")

print("All 15 classes are present in both splits.")
print("The train/test split is complete; no preprocessing has been fitted.")

,Total flows,Training flows,Test flows,Training percentage,Test percentage,Absolute difference (percentage points)
Label,,,,,,
BENIGN,2268391,1814712,453679,80.304062,80.304133,7.039894e-05
DoS Hulk,229964,183971,45993,8.141027,8.141060,3.344402e-05
PortScan,158804,127043,31761,5.621867,5.621903,3.678832e-05
DDoS,128006,102405,25601,4.531594,4.531543,5.026754e-05
DoS GoldenEye,10288,8230,2058,0.364191,0.364279,8.801977e-05
FTP-Patator,7931,6345,1586,0.280777,0.280732,4.462437e-05
SSH-Patator,5895,4716,1179,0.208691,0.208691,2.770474e-07
DoS slowloris,5796,4637,1159,0.205195,0.205151,4.452403e-05
DoS Slowhttptest,5499,4399,1100,0.194663,0.194707,4.399320e-05


All 15 classes are present in both splits.
The train/test split is complete; no preprocessing has been fitted.
